<center>
<h1> TP-Projet d'optimisation numérique </h1>
<h1> Algorithme des régions de confiance </h1>
</center>

# Régions de confiance avec Pas de Cauchy 

## Implémentation 

1. Coder l'algorithme du pas de Cauchy dans le fichier `src/cauchy.jl`). La spécification de cet algorithme est donnée dans le fichier.
2. Ecrire des tests exhaustifs (qui testent tous les cas de figure possibles) pour votre algorithme du pas de Cauchy. Vous remplirez pour cela le fichier `test/tester_cauchy.jl` sur le modèle des autres fichiers de tests et vous exécuterez dans la cellule de code ci-après ces tests.

In [38]:
include("../src/cauchy.jl")         # votre algorithme
include("../test/tester_cauchy.jl") # la fonction pour tester votre algorithme

tester_cauchy(cauchy); # tester l'algorithme

Test Summary: | Pass  Total  Time
Pas de Cauchy |    4      4  0.0s


3. Coder l'algorithme des régions de confiance (fichier `src/regions_de_confiance.jl`). Sa spécification est donnée dans le fichier.
4. Vérifier que les tests ci-dessous passent.

In [39]:
include("../src/regions_de_confiance.jl")
include("../test/tester_rc_cauchy.jl")

afficher = false # si true, alors affiche les résultats des algorithmes

tester_rc_cauchy(regions_de_confiance, afficher);

Affichage des résultats des algorithmes : false

Test Summary:        | Pass  Total  Time
RC et pas de Cauchy  |   15     15  0.5s


## Interprétation

<!-- Pour ces questions, des représentations graphiques sont attendues pour corroborer vos réponses. -->

1. Soit la fonction $f_1 \colon \mathbb{R}^3 \to \mathbb{R}$ définie par
$$
    f_1(x_1,x_2, x_3) = 2 (x_1 +x_2 + x_3 -3)^2 + (x_1-x_2)^2 + (x_2 - x_3)^2
$$
Quelle relation lie la fonction $f_1$ et son modèle de Taylor à l’ordre 2 ? Comparer alors les performances de l'algorithme de Newton et celui des régions de confiance avec le pas de Cauchy sur cette fonction.

$f_1$ est une fonction quadratique, donc son modèle de Taylor à l'ordre 2 lui est égal. Newton converge donc en une itération (cf [newton.ipynb](./newton.ipynb)).
L'algorithme du pas de Cauchy converge petit-à-petit. C'est-à-dire que le nombre d'itérations augmente quand $x_0$ s'éloigne de la solution.

In [40]:
using Plots
include("../src/regions_de_confiance.jl")
include("../src/newton.jl")
include("../test/tester_rc_cauchy.jl")

# Fonction f1
# -----------
f1(x) = 2 * (x[1] + x[2] + x[3] - 3)^2 + (x[1] - x[2])^2 + (x[2] - x[3])^2
grad_f1(x) = [4 * (x[1] + x[2] + x[3] - 3) + 2 * (x[1] - x[2]); # dérivée /x1
              4 * (x[1] + x[2] + x[3] - 3) - 2 * (x[1] - x[2]) + 2 * (x[2] - x[3]); # dérivée /x2
              4 * (x[1] + x[2] + x[3] - 3) - 2 * (x[2] - x[3])] # dérivée /x3
hess_f1(x) = [6 2 4; 2 8 2; 4 2 6]
solution = [1; 1; 1]

x0 = [0; 0; 0]
# avec régions de confiance (pas de Cauchy)
x_sol, f_sol, flag, nb_iters, xs = regions_de_confiance(f1, grad_f1, hess_f1, x0, algo_pas="cauchy")
afficher_resultats("Régions de confiance - Cauchy", "f1", x0, x_sol, f_sol, flag, nb_iters, solution)
# avec Newton
x_sol, f_sol, flag, nb_iters, xs = newton(f1, grad_f1, hess_f1, x0)
afficher_resultats("Newton", "f1", x0, x_sol, f_sol, flag, nb_iters, solution)

x0 = [10; 10; 10]
# avec régions de confiance (pas de Cauchy)
x_sol, f_sol, flag, nb_iters, xs = regions_de_confiance(f1, grad_f1, hess_f1, x0, algo_pas="cauchy")
# afficher_resultats("Régions de confiance - Cauchy", "f1", x0, x_sol, f_sol, flag, nb_iters, solution)
norm_diff_rc = [norm(x - solution) for x in xs]
# avec Newton
x_sol, f_sol, flag, nb_iters, xs = newton(f1, grad_f1, hess_f1, x0)
# afficher_resultats("Newton", "f1", x0, x_sol, f_sol, flag, nb_iters, solution)
norm_diff_newton = [norm(x - solution) for x in xs]
p1 = plot(norm_diff_rc, label="Régions de confiance - Cauchy", title="x0 = [10; 10; 10]", xlabel="Itérations (k)", ylabel="∥xk - x*∥")
plot!(p1, norm_diff_newton, label="Newton")
display(p1)

x0 = [100; 100; 100]
# avec régions de confiance (pas de Cauchy)
x_sol, f_sol, flag, nb_iters, xs = regions_de_confiance(f1, grad_f1, hess_f1, x0, algo_pas="cauchy")
norm_diff_rc = [norm(x - solution) for x in xs]
# avec Newton
x_sol, f_sol, flag, nb_iters, xs = newton(f1, grad_f1, hess_f1, x0)
norm_diff_newton = [norm(x - solution) for x in xs]
p2 = plot(norm_diff_rc, label="Régions de confiance - Cauchy", title="x0 = [100; 100; 100]", xlabel="Itérations (k)", ylabel="∥xk - x*∥")
plot!(p2, norm_diff_newton, label="Newton")
display(p2)

-------------------------------------------------------------------------
Résultats de : Régions de confiance - Cauchy appliqué à f1:
  * x0       = [0, 0, 0]
  * x_sol    = [1.0, 1.0, 1.0]
  * f(x_sol) = 0.0
  * nb_iters = 1
  * flag     = 0
  * solution = [1, 1, 1]
-------------------------------------------------------------------------
Résultats de : Newton appliqué à f1:
  * x0       = [0, 0, 0]
  * x_sol    = [1.0, 1.0, 0.9999999999999999]
  * f(x_sol) = 1.232595164407831e-32
  * nb_iters = 1
  * flag     = 0
  * solution = [1, 1, 1]


LoadError: syntax: missing comma or ) in argument list

2. Le rayon initial de la région de confiance est un paramètre important dans l’analyse
de la performance de l’algorithme. Sur quel(s) autre(s) paramètre(s) peut-on jouer
pour essayer d’améliorer cette performance ? Étudier l’influence d’au moins deux de
ces paramètres. Pour cela vous ferez des tests numériques et donnerez les résultats sous forme de tableaux et de graphiques.

In [41]:
# Expérimentations numériques à faire ici
# Vous pouvez utiliser le package Plots pour les affichages de courbes: using Plots

# Régions de confiance avec gradient conjugué tronqué

## Implémentation 

1. Implémenter l’algorithme du gradient conjugué tronqué (fichier `src/gct.jl`). Sa spécification est dans le fichier.
2. Vérifier que les tests ci-dessous passent.

In [42]:
include("../src/gct.jl")
include("../test/tester_gct.jl")

#
tester_gct(gct);

Test Summary:             | Pass  Total  Time
Gradient conjugué tronqué |    9      9  0.0s


3. Intégrer l’algorithme du gradient conjugué tronqué dans le code des régions de confiance.
4. Vérifier que les tests ci-dessous passent.

In [43]:
include("../src/regions_de_confiance.jl")
include("../test/tester_rc_gct.jl")

#
afficher = false # si true, alors affiche les résultats des algorithmes

#
tester_rc_gct(regions_de_confiance, afficher);

Affichage des résultats des algorithmes : false

Test Summary: | Pass  Total  Time
RC et gct     |   15     15  0.6s


## Interprétation  

Nous proposons de comparer l'utilisation du pas de Cauchy avec celle du gradient conjugué tronqué dans l'algorithme des régions de confiance.

**Remarques.**
* Nous vous demandons de réaliser des expérimentations numériques pour les comparaisons demandées ci-après.
* Vous devez utiliser l'argument optionnel `max_iter_gct` et la sortie `xs` de l'algorithme des régions de confiance.
* Vous pouvez comparer l'écart en norme entre les itérés de l'algorithme et la solution du problème.
* Vous trouverez des choses utiles dans le fichier `test/fonctions_de_tests.jl`.

1. Comparer dans le cas où l'on force le gradient conjugué tronqué à ne faire qu'une seule itération. Que remarquez vous ?
2. Comparer dans le cas général. Que remarquez vous ?
3. Quels sont les avantages et inconvénients des deux approches ?

In [44]:
# Expérimentations numériques à faire ici.
# Vous pouvez utiliser le package Plots pour les affichages de courbes: using Plots